# Øvelser: Fra rå fil til document-term matrix

**Social Data Science 1 — lektion 6**

I sidste uge hentede I data ned fra nettet. I dag arbejder vi med resultatet: 7.207
jobopslag fra fire portaler, indsamlet med fire forskellige scrapere i efteråret 2023.

Målet er ikke at analysere dem. Målet er at nå frem til en **datastruktur**, man kan
analysere — og at kunne gøre rede for hvert eneste valg undervejs.

**Fil:** `jobapplications_fall-2023.csv`

**Sådan arbejder I:**

- Skriv hvert valg ned. Alle sammen. De skal bruges i portfolien.
- Gæt tallene, før I kører koden.
- Læs fejlbeskeder. De første fire opgaver består næsten kun af fejlbeskeder.


---

## Opgave 1: Få filen ind

**Trin 1 — prøv det oplagte.** Kør cellen og læs fejlen.


In [ ]:
import pandas as pd

opslag = pd.read_csv("jobapplications_fall-2023.csv")


**Trin 2 — separatoren.**

Filen er lavet i et dansk regneark, hvor kommaet er decimaltegn. Derfor er
kolonnerne adskilt af noget andet.

Åbn filen i en teksteditor og kig på den første linje. Tilføj det rigtige `sep=`.


In [ ]:
# Din kode her



**Trin 3 — overskriften.**

Nu kommer den ind, men kolonnerne hedder noget forkert. Kig på de første to rækker.

*Hint:* `skiprows=`


In [ ]:
# Din kode her



**Trin 4 — tegnsættet.**

Print et par titler. Ser `København` rigtig ud?

En tekstfil er bytes. Tegnsættet er aftalen om, hvilket tegn hver byte betyder, og
filen indeholder ingen oplysning om, hvilken aftale der gælder. Man gætter.

Prøv `encoding=` med `"utf-8"`, `"latin-1"`, `"cp1252"` og `"mac_roman"`.
Hvilken giver rigtige danske bogstaver?


In [ ]:
# Din kode her



### 1.5 Til diskussion

1. Tre argumenter til `read_csv`. Hver af dem svarer til noget, en scraper gjorde for
   et år siden. Hvad ville I have gjort anderledes, hvis I selv havde skrevet den?
2. Hvad sker der, hvis I bruger det forkerte tegnsæt og ikke opdager det? Hvor i
   analysen ville det dukke op?


---

## Opgave 2: Metadata er også tekst

**Trin 1 — hvad mangler?**

Kør `opslag.isna().sum()`. Kolonnen `company` mangler for de fleste opslag.

Undersøg om det er tilfældigt. Grupper efter `source` og se hvor mange der mangler
i hver kilde.


In [ ]:
# Din kode her



**Trin 2 — find firmanavnet.**

Det er ikke væk. Print `various_meta1` for et opslag, hvor `company` mangler.

Hvad står der, og hvordan er felterne adskilt?


In [ ]:
# Din kode her



**Trin 3 — hent det ud.**

Del feltet op med `.split("\n")` — samme metode som i lektion 2, bare med
linjeskift i stedet for mellemrum.

Byg en ny kolonne `firma`, der indeholder den første linje.

*Hint:* `opslag["various_meta1"].str.split("\n").str[0]`


In [ ]:
# Din kode her



**Trin 4 — fælden.**

Tæl, hvor mange linjer der er i hvert felt:

```python
opslag["various_meta1"].str.split("\n").str.len().value_counts()
```

De fleste har tre. Nogle har to. Kig på et par af dem med to linjer.

Hvad er der havnet i jeres `firma`-kolonne for netop de opslag? Og hvordan finder I
dem uden at læse alle 7.207 igennem?


In [ ]:
# Din kode her



**Trin 5 — datoen.**

Tredje linje ser ud som `Indrykket 24-10-2023`. Hent datoen ud og gør den til en
rigtig dato.

Hvor mange lykkes det for? Og hvilke kilder har slet ingen dato?


In [ ]:
# Din kode her



### 2.6 Til diskussion

1. Er det bedre at rette i data eller at skrive kode, der kan håndtere alle fire
   scraperes formater?
2. Hvad skal der stå i metodeafsnittet om, at firmanavnet er hentet ud af et
   fritekstfelt for 87 % af opslagene?


---

## Opgave 3: Byg preprocessing-kæden

Nu til selve oprydningen. Efter hvert trin skal I notere **to** tal: antal tokens og
antal typer. Brug papirarket.

**Trin 0 — udgangspunktet.**

Gæt begge tal, før I kører cellen.


In [ ]:
tekster = opslag["job_description"].dropna().astype(str)

alle = [o for t in tekster for o in t.split()]

print("Tokens:", len(alle))
print("Typer: ", len(set(alle)))


**Trin 1 — små bogstaver.**

Gæt først: hvilket af de to tal ændrer sig?


In [ ]:
# Din kode her



**Trin 2 — tegnsætning.**

```python
import string
tegn = string.punctuation + "«»…–—"
```

Brug `.strip(tegn)` på hvert ord, og smid tomme strenge væk bagefter.

Hvorfor bliver `søger/finder` stående som ét ord?


In [ ]:
# Din kode her



**Trin 3 — tal.**

Fjern alle ord, der indeholder et ciffer.

*Hint:* `any(c.isdigit() for c in o)`

Kig på ti af de ord, I lige har fjernet. Var de alle sammen støj?


In [ ]:
# Din kode her



**Trin 4 — stopord.**


In [ ]:
stopord = set("""og i jeg det at en den til er som på de med han af for ikke
der var mig sig men et har om vi min havde ham hun nu over da fra du ud sin
dem os op man hans hvor eller hvad skal selv her alle vil blev kunne ind når
være dog jo dette dig deres end mit også under have dens hvis dine disse
hvem vores jer sådan andre nogle bliver blive kan""".split())

# Din kode her


**Trin 5 — sjældne ord.**

Tæl for hvert ord, i hvor mange **dokumenter** det optræder. Behold kun dem, der
findes i mindst fem.

```python
from collections import Counter

df = Counter()
for t in tekster:
    df.update(set(...))     # hvorfor set() og ikke bare listen?
```


In [ ]:
# Din kode her



### 3.6 Til diskussion

1. To trin fjerner mange tokens og næsten ingen typer. To gør det modsatte. Hvilke?
2. Hvor mange ord optræder i præcis ét dokument? Hvad kan de bruges til, hvis man
   vil sammenligne dokumenter?
3. Hvor stor en del af det oprindelige vokabular er tilbage?


---

## Opgave 4: Rækkefølgen og kombinationen

Denny & Spirlings hovedpointe: syv operationer, der hver kan slås til eller fra,
giver 128 forskellige udgaver af det samme materiale.

**Trin 1 — gør kæden til en funktion.**

```python
def rens(tekst, lower=True, punkt=True, tal=True, stop=True):
    toks = tekst.split()
    if lower: ...
    return toks
```

Fire argumenter, der kan slås til og fra.


In [ ]:
# Din kode her



**Trin 2 — kør alle seksten kombinationer.**

```python
import itertools

for L, P, N, S in itertools.product([True, False], repeat=4):
    ...
```

Brug de første 2.000 opslag, så det ikke tager for lang tid. Print vokabularets
størrelse for hver kombination.

Hvor stor er forskellen mellem den mindste og den største?


In [ ]:
# Din kode her



**Trin 3 — rækkefølgen.**

Fjern stopord **før** I laver små bogstaver, i stedet for efter.

Får I det samme vokabular? Hvorfor / hvorfor ikke?


In [ ]:
# Din kode her



### 4.4 Til diskussion

1. Hvis I lavede en klyngeanalyse på hver af de seksten, ville I så få de samme
   klynger?
2. Ved klassifikation kan man måle sig frem til den bedste kombination. Hvorfor kan
   man ikke det ved klyngeanalyse?
3. Hvad ville I skrive i metodeafsnittet, hvis resultatet holdt på tværs af alle
   seksten? Og hvis det ikke gjorde?


---

## Opgave 5: Frekvens og hvad den ikke viser

**Trin 1 — de hyppigste ord.**

Tæl de tyve hyppigste ord i det rensede materiale.

Kig på listen, før I læser videre.


In [ ]:
# Din kode her



**Trin 2 — sproget.**

Hvor mange af opslagene er på engelsk? Et groft mål:

```python
paa_engelsk = tekster.str.lower().str.count(r"\bthe\b") >= 10
```

Hvad gør en dansk stopordsliste ved engelske stopord?

Er det et forbehandlingsproblem eller et korpusproblem?


In [ ]:
# Din kode her



**Trin 3 — de domænespecifikke ord.**

Ser man bort fra de engelske, står ord som `arbejde`, `stillingen` og `opgaver`
tilbage blandt de hyppigste.

De står i næsten alle jobopslag, netop fordi det er jobopslag. Byg en
**korpusspecifik** stopordsliste: find de ord, der optræder i over 80 % af
opslagene, og se hvad der bliver tilbage.


In [ ]:
# Din kode her



**Trin 4 — nøgleord.**

Sammenlign to kilder i stedet for at kigge på hele materiale under ét.

Hvilke ord er relativt hyppigere i `job.aka.dk` end i `jobbank.dk`?

*Hint:* beregn hvert ords andel af tokens i hver kilde, og se på forholdet mellem
de to andele.


In [ ]:
# Din kode her



---

## Opgave 6: Document-term matrix

**Trin 1 — byg den.**

Én række per opslag, én kolonne per ord, tællinger i cellerne.

Byg den i hånden med en liste af dictionaries — samme mønster som i lektion 3.

```python
raekker = []
for t in tekster:
    raekker.append(Counter(rens(t)))

dtm = pd.DataFrame(raekker).fillna(0)
```

Kør den på 500 opslag først. Hvor mange rækker og kolonner får I?


In [ ]:
# Din kode her



**Trin 2 — sparsity.**

Hvor stor en andel af cellerne er nul?

*Hint:* `(dtm == 0).sum().sum() / dtm.size`


In [ ]:
# Din kode her



**Trin 3 — hvad det betyder.**

Vælg to tilfældige opslag. Hvor mange ord har de til fælles?

Gør det ti gange. Hvad er typisk?


In [ ]:
# Din kode her



### 6.4 Til diskussion

1. Hvert opslag er et punkt i et rum med tusindvis af dimensioner. Hvad betyder
   "afstand" mellem to punkter der?
2. Hvis næsten alle par af opslag har nul fælles ord, hvad sker der så med en
   klyngeanalyse?
3. Hvad tror I PCA gør ved det problem? (Vi tager det næste gang.)


---

## Opgave 7: spaCy

Denne del skal køres i UCloud eller lokalt.

```bash
pip install spacy
python -m spacy download da_core_news_sm
```

**Trin 1 — kør pipelinen på én tekst.**

Slidesene viste jer koden, ikke resultatet. Kør den og skriv ned, hvad I får.


In [ ]:
import spacy

nlp = spacy.load("da_core_news_sm")
doc = nlp("Vi søger en erfaren socialrådgiver til vores team i Aarhus.")

for tok in doc:
    print(tok.text, tok.lemma_, tok.pos_, tok.is_stop)


**Trin 2 — stemming mod lemmatisering.**

Kør pipelinen på ordene `søger`, `søgte`, `søgt`, `universitet`, `universel`.

Hvad bliver de til? Hvad ville en simpel regel — klip de sidste to bogstaver af —
have gjort ved de samme ord?


In [ ]:
# Din kode her



**Trin 3 — ordklasser som filter.**

Kør pipelinen på 200 opslag og behold kun navneord og tillægsord.

```python
docs = nlp.pipe(tekster[:200], batch_size=50)
```

Sammenlign vokabularet med det, I fik af regelversionen på de samme 200 opslag.
Hvor stor er forskellen?


In [ ]:
# Din kode her



**Trin 4 — find en fejl.**

Modellen er trænet på dansk nyhedstekst. Jobopslag er ikke nyhedstekst, og en del af
materialet er på engelsk.

Find mindst tre steder, hvor lemmatiseringen eller ordklassen er forkert. Hvilken
slags ord går det galt på?


In [ ]:
# Din kode her



### 7.5 Til diskussion

1. Reglerne kørte på sekunder. spaCy tager minutter. Hvornår er det pengene værd?
2. I kan læse jeres egen stopordsliste. I kan ikke læse en model. Betyder det noget
   for, hvad I kan skrive i metodeafsnittet?


---

## Opgave 8: Til portfolien

Skriv en preprocessing-protokol for jeres eget materiale.

1. **Hvert valg, i rækkefølge.** Hvad gjorde I, og hvorfor netop det?

2. **Hvad kostede hvert valg?** Ét konkret eksempel per trin på noget, der gik tabt.

3. **Tokens og typer** før og efter. Hvor stor blev reduktionen?

4. **Robusthed.** Prøvede I flere kombinationer? Holdt resultatet?

Protokollen skal være detaljeret nok til, at en anden kan gentage den præcist.
Det er hele pointen i Denny & Spirlings artikel.


In [ ]:
# Din kode her



---

## Ekstraopgaver

**A. n-grams.** Byg bigrams i stedet for enkeltord. Hvor stort bliver vokabularet?
Find fem bigrams, der betyder noget andet end de to ord hver for sig.

**B. Ordlængde.** Er der forskel på, hvor lange opslagene er på tværs af de fire
kilder? Tegn det med plotnine. Hvad betyder det for optællinger per dokument?

**C. Dubletter.** Er der opslag, der er slået op flere gange? Hvordan finder I dem,
og hvad betyder de for en klyngeanalyse?

**D. tf-idf.** Slå formlen op og beregn den i hånden for tre ord i tre opslag.
Hvilke ord vægtes op, og hvilke ned?

**E. Konkordans.** Vælg et ord, der optræder ofte. Print de fem ord før og efter
hver forekomst. Hvad kan I se, som optællingen ikke viste?


In [ ]:
# Din kode her

